In [1]:
# --- SAFETY FLAG: NO HARDWARE MODE ---
SIMULATION_MODE = True

print("[BOOT] Segmentation System Dry Run")
print("[BOOT] SIMULATION MODE =", SIMULATION_MODE)

import json
from pathlib import Path

[BOOT] Segmentation System Dry Run
[BOOT] SIMULATION MODE = True


In [2]:
import sys
from pathlib import Path

# Find repo root dynamically (go up until we find Core/)
repo_root = Path.cwd()

while not (repo_root / "Core").exists():
    repo_root = repo_root.parent

sys.path.append(str(repo_root))

print("[BOOT] Repo root detected at:", repo_root)
print("[BOOT] sys.path updated")

[BOOT] Repo root detected at: C:\Users\40299205\Documents\VictorFlow
[BOOT] sys.path updated


In [3]:
# Core system imports
from Core.experiment_context import ExperimentManager
from Ecosystems.segmentation import Segmentation

print("[BOOT] Core modules loaded")

[BOOT] Core modules loaded


In [4]:
class TraceLogger:
    def log(self, msg):
        print(f"[TRACE] {msg}")

trace = TraceLogger()

print("[BOOT] Trace logger active")

[BOOT] Trace logger active


In [5]:
manager = ExperimentManager()

experiment_id = "vb_1_13_dummy"

manager.load_experiment(experiment_id)

print("\n[STATE]")
manager.status()

[EXPERIMENT] vb_1_13_dummy loaded
[EXPERIMENT] Plan ready
[EXPERIMENT] Log ready
[EXPERIMENT] State = prepared
[EXPERIMENT] Awaiting arm_experiment()

[STATE]
Mode: experiment
Experiment: vb_1_13_dummy
State: prepared
Slug index: 0


In [6]:
manager.arm_experiment()

print("\n[STATE]")
manager.status()

[EXPERIMENT] vb_1_13_dummy armed
[EXPERIMENT] Awaiting execute_experiment()

[STATE]
Mode: experiment
Experiment: vb_1_13_dummy
State: armed
Slug index: 0


In [7]:
class SimulatedSegmentation(Segmentation):

    def __init__(self, *args, **kwargs):
        kwargs["sim_mode"] = True
        super().__init__(*args, **kwargs)

    # override pump + valve actions with prints only
    def _simulate(self, action):
        print(f"[SIM] {action}")

    def prime_gas_path(self, duration_s):
        self._simulate(f"FSM → GAS_TO_DIM | DIM → INJECT | gas_prime_s={duration_s}")
        super().prime_gas_path(duration_s)

    def launch_segment(self, flowrate_ul_min):
        self._simulate(f"START FLOW @ {flowrate_ul_min} uL/min")
        super().launch_segment(flowrate_ul_min)

print("[BOOT] Simulated Segmentation ready")

[BOOT] Simulated Segmentation ready


In [8]:
# ============================================================
# SIMULATION HARDWARE LAYER (NO REAL INSTRUMENTS)
# ============================================================

class DummyCarrierPump:
    def set_flow_rate(self, rate):
        print(f"[SIM PUMP] set_flow_rate({rate})")

    def start_flow(self):
        print("[SIM PUMP] start_flow()")

    def stop_flow(self):
        print("[SIM PUMP] stop_flow()")


class DummyFSM:
    def carrier_to_dim(self):
        print("[SIM FSM] carrier_to_dim()")

    def gas_to_dim(self):
        print("[SIM FSM] gas_to_dim()")


class DummyDIM:
    def inject(self):
        print("[SIM DIM] inject()")

    def load(self):
        print("[SIM DIM] load()")


class DummyRSG:
    def build_rxn_segment(self, slug_plan, air_gap_between=5.0):
        print(f"[SIM RSG] build_rxn_segment({slug_plan.get('slug_id', 'no-id')})")

        reaction_plan = slug_plan.get("reaction_plan", [])
        total_volume = sum(
            c.get("volume_uL", c.get("volume", 0)) for c in reaction_plan
        )

        return {
            "total_volume_ul": total_volume,
            "num_components": len(reaction_plan),
        }

    def dispense_in_dim(self, volume):
        print(f"[SIM RSG] dispense_in_dim({volume} uL)")


# ============================================================
# INSTANTIATE SIM SYSTEM
# ============================================================

dim = DummyDIM()
fsm = DummyFSM()
carrier = DummyCarrierPump()
rsg = DummyRSG()

seg = Segmentation(
    dim=dim,
    fsm=fsm,
    carrier_pump=carrier,
    rsg=rsg,
    sim_mode=True
)

print("[BOOT] Segmentation system instantiated (SIM)")

[SIM FSM] carrier_to_dim()
[SIM DIM] inject()
[BOOT] Segmentation system instantiated (SIM)


In [9]:
manager.preview()


[EXPERIMENT PREVIEW]
------------------------------------------------------------
1. vb_1_13_dummy_slug_01 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
2. vb_1_13_dummy_slug_02 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
3. vb_1_13_dummy_slug_03 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
4. vb_1_13_dummy_slug_04 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
5. vb_1_13_dummy_slug_05 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
6. vb_1_13_dummy_slug_06 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
7. vb_1_13_dummy_slug_07 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
8. vb_1_13_dummy_slug_08 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
9. vb_1_13_dummy_slug_09 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
10. vb_1_13_dummy_slug_10 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
11. vb_1_13_dummy_slug_11 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
12. vb_1_13_dummy_slug_12 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
13. vb_1_13_dummy_slug_13 | 1 componen

[{'slug_number': 1,
  'slug_id': 'vb_1_13_dummy_slug_01',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_number': 2,
  'slug_id': 'vb_1_13_dummy_slug_02',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_number': 3,
  'slug_id': 'vb_1_13_dummy_slug_03',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_number': 4,
  'slug_id': 'vb_1_13_dummy_slug_04',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_number': 5,
  'slug_id': 'vb_1_13_dummy_slug_05',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_number': 6,
  'slug_id': 'vb_1_13_dummy_slug_06',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'rack1:1 (100.0 uL)'},
 {'slug_number': 7,
  'slug_id': 'vb_1_13_dummy_slug_07',
  'num_components': 1,
  'total_volume_uL': 100.0,
  '

In [10]:
results = manager.execute_experiment(
    seg=seg,
    gas_prime_s=2.0,
    flowrate_ul_min=500,
)

print("\n[RESULTS]")
print(json.dumps(results, indent=2))

[EXPERIMENT] Executing vb_1_13_dummy
[SIM FSM] gas_to_dim()
[SIM DIM] inject()
[Segmentation] Phase: READY -> GAS_PRIMED
[Segmentation] Phase: GAS_PRIMED -> LOOP_LOADING
[SIM DIM] load()
[SIM RSG] build_rxn_segment(vb_1_13_dummy_slug_01)
[Segmentation] Phase: LOOP_LOADING -> LOOP_LOADED
[SIM DIM] inject()
[SIM FSM] carrier_to_dim()
[SIM PUMP] set_flow_rate(500)
[SIM PUMP] start_flow()
[Segmentation] Phase: LOOP_LOADED -> RUNNING
[Segmentation] Segment launched at 500 µL/min
[EXPERIMENT] Completed vb_1_13_dummy_slug_01 (100.0 uL)
[SIM FSM] gas_to_dim()
[SIM DIM] inject()
[Segmentation] Phase: RUNNING -> GAS_PRIMED
[Segmentation] Phase: GAS_PRIMED -> LOOP_LOADING
[SIM DIM] load()
[SIM RSG] build_rxn_segment(vb_1_13_dummy_slug_02)
[Segmentation] Phase: LOOP_LOADING -> LOOP_LOADED
[SIM DIM] inject()
[SIM FSM] carrier_to_dim()
[SIM PUMP] set_flow_rate(500)
[SIM PUMP] start_flow()
[Segmentation] Phase: LOOP_LOADED -> RUNNING
[Segmentation] Segment launched at 500 µL/min
[EXPERIMENT] Complete

In [11]:
manager.status()

print("\n[LOG FILE]")
with open(manager.context.log_path, "r") as f:
    log = json.load(f)

log

Mode: experiment
Experiment: vb_1_13_dummy
State: completed
Slug index: 15

[LOG FILE]


{'experiment_id': 'vb_1_13_dummy',
 'created_at': '2026-04-28T09:38:13',
 'events': [{'timestamp': '2026-04-28T09:55:28',
   'event_type': 'begin_run',
   'payload': {'experiment_id': 'vb_1_13_dummy'}},
  {'timestamp': '2026-04-28T12:38:16',
   'event_type': 'experiment_loaded',
   'payload': {'experiment_id': 'vb_1_13_dummy'}},
  {'timestamp': '2026-04-28T12:38:22',
   'event_type': 'experiment_armed',
   'payload': {'experiment_id': 'vb_1_13_dummy'}},
  {'timestamp': '2026-04-28T12:43:06',
   'event_type': 'experiment_loaded',
   'payload': {'experiment_id': 'vb_1_13_dummy'}},
  {'timestamp': '2026-04-28T12:43:12',
   'event_type': 'experiment_armed',
   'payload': {'experiment_id': 'vb_1_13_dummy'}},
  {'timestamp': '2026-04-28T12:47:45',
   'event_type': 'experiment_loaded',
   'payload': {'experiment_id': 'vb_1_13_dummy'}},
  {'timestamp': '2026-04-28T12:47:46',
   'event_type': 'experiment_armed',
   'payload': {'experiment_id': 'vb_1_13_dummy'}},
  {'timestamp': '2026-04-28T12:5

**Below is another dry run that includes the ExperimentValidator class (in experiment_validator.py in Core)**

In [1]:
import sys
from pathlib import Path
import json

# Find repo root dynamically (go up until we find Core/)
repo_root = Path.cwd()

while not (repo_root / "Core").exists():
    repo_root = repo_root.parent

sys.path.append(str(repo_root))

print("[BOOT] Repo root detected at:", repo_root)
print("[BOOT] sys.path updated")

[BOOT] Repo root detected at: C:\Users\40299205\Documents\VictorFlow
[BOOT] sys.path updated


In [2]:
from Core.experiment_builder import ExperimentBuilder
from Core.experiment_context import ExperimentManager
from Core.experiment_validator import ExperimentValidator

from Ecosystems.segmentation import Segmentation

print("[BOOT] Core modules loaded")

[BOOT] Core modules loaded


In [3]:
class SimDIM:
    def inject(self):
        print("[SIM DIM] inject()")

    def load(self):
        print("[SIM DIM] load()")


class SimFSM:
    def carrier_to_dim(self):
        print("[SIM FSM] carrier_to_dim()")

    def gas_to_dim(self):
        print("[SIM FSM] gas_to_dim()")


class SimPump:
    def set_flow_rate(self, x):
        print(f"[SIM PUMP] set_flow_rate({x})")

    def start_flow(self):
        print("[SIM PUMP] start_flow()")

    def stop_flow(self):
        print("[SIM PUMP] stop_flow()")


class SimRSG:
    def build_rxn_segment(self, slug_plan, air_gap_between=5.0):

        slug_id = slug_plan["slug_id"]

        total = sum(
            c["volume_uL"]
            for c in slug_plan["reaction_plan"]
        )

        print(f"[SIM RSG] build_rxn_segment({slug_id})")

        return {
            "slug_id": slug_id,
            "total_volume_ul": total,
            "num_components": len(slug_plan["reaction_plan"]),
        }

print("[BOOT] Sim hardware ready")

[BOOT] Sim hardware ready


In [4]:
dim = SimDIM()
fsm = SimFSM()
pump = SimPump()
rsg = SimRSG()

seg = Segmentation(
    dim=dim,
    fsm=fsm,
    carrier_pump=pump,
    rsg=rsg,
    sim_mode=True,
)

print("[BOOT] Segmentation instantiated (SIM)")
print(seg.state)

[SIM FSM] carrier_to_dim()
[SIM DIM] inject()
[BOOT] Segmentation instantiated (SIM)
Segmentation phase = READY | DIM = INJECT | FSM = CARRIER_TO_DIM


In [5]:
builder = ExperimentBuilder()

rows = []

for i in range(1, 6):
    rows.append(
        {
            "slug_id": f"dry_slug_{i:02d}",
            "module": "A",
            "vial": 1,
            "volume_uL": 100,
        }
    )

build = builder.build_and_create(
    experiment_id="seg_dry_run_demo_001",
    rows=rows,
    description="Dry run notebook validation experiment",
    global_conditions={
        "flowrate_ul_min": 500,
        "gas_prime_s": 2.0,
    },
    overwrite=True,
)

print("[BUILD COMPLETE]")
print(build["experiment_dir"])

[BUILD COMPLETE]
C:\Users\40299205\Documents\VictorFlow\Experiments\seg_dry_run_demo_001


In [6]:
manager = ExperimentManager()

manager.load_experiment("seg_dry_run_demo_001")
manager.status()

[EXPERIMENT] seg_dry_run_demo_001 loaded
[EXPERIMENT] Plan ready
[EXPERIMENT] Log ready
[EXPERIMENT] State = prepared
[EXPERIMENT] Awaiting arm_experiment()
Mode: experiment
Experiment: seg_dry_run_demo_001
State: prepared
Slug index: 0


In [7]:
validator = ExperimentValidator()

print("[BOOT] Validator ready")

[BOOT] Validator ready


In [8]:
plan_report = validator.validate_plan(manager.context.plan)

print("[PLAN VALIDATION]")
print(json.dumps(plan_report, indent=2))

[PLAN VALIDATION]
{
  "passed": true,
  "errors": [],
  "warnings": [],
  "slug_count": 5
}


In [9]:
manager.arm_experiment()
manager.status()

[EXPERIMENT] seg_dry_run_demo_001 armed
[EXPERIMENT] Awaiting execute_experiment()
Mode: experiment
Experiment: seg_dry_run_demo_001
State: armed
Slug index: 0


In [10]:
preflight = validator.validate_preflight(manager, seg)

print("[PREFLIGHT]")
print(json.dumps(preflight, indent=2))

[PREFLIGHT]
{
  "passed": true,
  "errors": []
}


In [11]:
manager.preview()


[EXPERIMENT PREVIEW]
------------------------------------------------------------
1. dry_slug_01 | 1 components | 100.0 uL
   A:1 (100.0 uL)
2. dry_slug_02 | 1 components | 100.0 uL
   A:1 (100.0 uL)
3. dry_slug_03 | 1 components | 100.0 uL
   A:1 (100.0 uL)
4. dry_slug_04 | 1 components | 100.0 uL
   A:1 (100.0 uL)
5. dry_slug_05 | 1 components | 100.0 uL
   A:1 (100.0 uL)
------------------------------------------------------------
[EXPERIMENT] 5 slugs total



[{'slug_number': 1,
  'slug_id': 'dry_slug_01',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'A:1 (100.0 uL)'},
 {'slug_number': 2,
  'slug_id': 'dry_slug_02',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'A:1 (100.0 uL)'},
 {'slug_number': 3,
  'slug_id': 'dry_slug_03',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'A:1 (100.0 uL)'},
 {'slug_number': 4,
  'slug_id': 'dry_slug_04',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'A:1 (100.0 uL)'},
 {'slug_number': 5,
  'slug_id': 'dry_slug_05',
  'num_components': 1,
  'total_volume_uL': 100.0,
  'components': 'A:1 (100.0 uL)'}]

In [12]:
results = manager.execute_experiment(seg=seg)

print("\n[RESULTS]")
print(json.dumps(results, indent=2))

[EXPERIMENT] Executing seg_dry_run_demo_001
[SIM FSM] gas_to_dim()
[SIM DIM] inject()
[Segmentation] Phase: READY -> GAS_PRIMED
[Segmentation] Phase: GAS_PRIMED -> LOOP_LOADING
[SIM DIM] load()
[SIM RSG] build_rxn_segment(dry_slug_01)
[Segmentation] Phase: LOOP_LOADING -> LOOP_LOADED
[SIM DIM] inject()
[SIM FSM] carrier_to_dim()
[SIM PUMP] set_flow_rate(500.0)
[SIM PUMP] start_flow()
[Segmentation] Phase: LOOP_LOADED -> RUNNING
[Segmentation] Segment launched at 500.0 µL/min
[EXPERIMENT] Completed dry_slug_01 (100.0 uL)
[SIM FSM] gas_to_dim()
[SIM DIM] inject()
[Segmentation] Phase: RUNNING -> GAS_PRIMED
[Segmentation] Phase: GAS_PRIMED -> LOOP_LOADING
[SIM DIM] load()
[SIM RSG] build_rxn_segment(dry_slug_02)
[Segmentation] Phase: LOOP_LOADING -> LOOP_LOADED
[SIM DIM] inject()
[SIM FSM] carrier_to_dim()
[SIM PUMP] set_flow_rate(500.0)
[SIM PUMP] start_flow()
[Segmentation] Phase: LOOP_LOADED -> RUNNING
[Segmentation] Segment launched at 500.0 µL/min
[EXPERIMENT] Completed dry_slug_02 (

In [13]:
post = validator.validate_post_run(manager, results)

print("[POST RUN]")
print(json.dumps(post, indent=2))

[POST RUN]
{
  "passed": true,
  "errors": [],
  "warnings": [],
  "planned_slugs": 5,
  "executed_slugs": 5
}


In [14]:
log_report = validator.validate_log(manager.context.log_path)

print("[LOG VALIDATION]")
print(json.dumps(log_report, indent=2))

[LOG VALIDATION]
{
  "passed": true,
  "errors": [],
  "event_count": 8
}


In [15]:
all_passed = all([
    plan_report["passed"],
    preflight["passed"],
    post["passed"],
    log_report["passed"],
])

print("\n======================")
print("SYSTEM SUMMARY")
print("======================")

if all_passed:
    print("PASS")
else:
    print("FAIL")


SYSTEM SUMMARY
PASS
